[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tonyscan6003/CE5021_group01_AY2627/blob/main/Assign01/Assignment1.ipynb)

## Working with this notebook on GitHub & Colab
This notebook is in your group's private GitHub repository. Each student saves their own work to a personal **branch** (their own copy of the repository's files) named after their GitHub username, so group members never overwrite each other's work.

**Once, before your first save: create your branch** (on github.com)
1. Open your group repository on GitHub.
2. Click the branch button above the file list (it shows `main`), type your GitHub username and click **Create branch: *username* from main**.

**Open the notebook** (in Colab)
1. Go to [colab.research.google.com](https://colab.research.google.com) and choose *File → Open notebook → GitHub*.
2. Tick **Include private repos**. The first time, Colab asks for access to your GitHub account: allow it.
3. Select your group repository, then the branch:
   * a new assignment: branch `main`
   * continuing your own work: **your branch**
4. Click the notebook to open it.

**Save your work** (Colab does not save to GitHub automatically)
* Choose *File → Save a copy in GitHub*: select your group repository and **your branch** (not `main`), keep the file path, and write a short commit message saying what you changed.
* Save regularly. Changes not saved to GitHub are lost when you close the Colab tab.
* Plots and printed results are saved with the notebook, so save again after your final training run.

# Regularisation and Optimisation
This assignment investigates training of CNNs on CIFAR with a goal of maximising accuracy.

[CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) is a dataset with small size (32x32) images and 50k training samples.
* It is often used as a "Hello World" type data set to setup a training pipeline. As it is relatively fast to train a network with CIFAR-10, this dataset is also widely used for running comparison or optimisation experiments. For example comparing different networks, regularisation methods or optimisers. (In pytorch the CIFAR-10 dataset is available as a [pytorch dataset](https://pytorch.org/vision/0.18/generated/torchvision.datasets.CIFAR10.html) simplifying loading the dataset.)

The starting point in this assignment is an untrained convolutional neural network.
* You will not modify the structure of the network. (We will discuss network structures in more detail later in the module).
* The network has sufficient capacity (Parameters/FLOPS) to achieve high accuracy.
* The notebook (starting configuration) will train the model without regularisation and using the SGD optimiser. From inspection of the training/validation accuracy/loss, it will be observed that the network is overfitting.
* You will improve the performance of the network using regularisation techniques (See below).
* The goal is to use one or several techniques to prevent overfitting and maximise test accuracy.

Regularisation techniques.
* [Data Augmentation using pytorch transforms](https://pytorch.org/vision/0.15/transforms.html)
* Regularisation layers such as [dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html) and [batch normalisation](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html).

The learning rate also acts as a regulariser, with large learning rates having a regularising effect. You may also include a LR schedule as detailed in the [pytorch optim package](https://pytorch.org/docs/stable/optim.html). The optim package contains several implementations of LR schedules including those detailed in the papers:
* [Cyclical Learning Rates for Training Neural Networks](https://arxiv.org/pdf/1506.01186)
* [Super-Convergence: Very Fast Training of Neural
Networks Using Large Learning Rates](https://arxiv.org/pdf/1708.07120)

How to approach this Assignment.
* There are a lot things to try and possible approaches. Keep a record of what you have tried (`history.summary()` prints the settings and final accuracies of each run). Apart from recording training and test accuracy, ensure that you record the key hyperparameters, including optimiser, base learning rate, number of epochs for each run. This makes repeating experiments possible if you want to try a variation.
* Your final submitted notebook should include your best attempt showing training curves. Post your results and comments on your approach to the module forum.
* Routine code (loading data, measuring accuracy, plotting) comes from the course helper package `ce5021`, installed in the first cell. The training loop is in this notebook so you can read and change it. To see how a helper works, run `help(evaluate)` or `evaluate??` in a cell. (Do not submit a notebook with multiple approaches)

## 1. Setup

The dependencies are installed in the local virtual environment. Select that environment as the notebook kernel before running the imports.


In [ ]:
import sys
print("Python environment:", sys.executable)


In [ ]:
import copy
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import v2
from torchinfo import summary
from sklearn.metrics import classification_report

from ce5021.data import cifar10_datasets, CIFAR10_CLASSES, IMAGENET_MEAN, IMAGENET_STD
from ce5021.train import get_device, evaluate, predict, History
from ce5021.plots import show_batch, show_predictions, plot_confusion_matrix


## 2. Dataset and experiment configuration

The original 50,000 training images are split into 45,000 training and 5,000 validation images using a fixed seed. Only the training subset receives training augmentation; validation and test images use deterministic conversion and normalisation. The 10,000 test images are reserved for final evaluation after choosing an approach.


In [ ]:
# config = dict(
#     run_name="baseline", n_epochs=20, lr=0.01,
#     momentum=0.9, batch_size=128,
# )

# Baseline with validation split:
# config = dict(
#     run_name="baseline_validation_split", augmentation="none",
#     n_epochs=20, lr=0.01, momentum=0.9, batch_size=128,
#     seed=42, validation_size=5_000,
# )

# Experiment 1: random crop and horizontal flip.
# config = dict(
#     run_name="random_crop_flip", augmentation="crop_flip",
#     n_epochs=20, lr=0.01, momentum=0.9, batch_size=128,
#     seed=42, validation_size=5_000,
# )

# Experiment 2: random crop and horizontal flip, 40 epochs.
# config = dict(
#     run_name="random_crop_flip_40", augmentation="crop_flip",
#     n_epochs=40, lr=0.01, momentum=0.9, batch_size=128,
#     seed=42, validation_size=5_000,
# )

# Experiment 3: lower the learning rate after epoch 30.
config = dict(
    run_name="crop_flip_step_lr", augmentation="crop_flip",
    n_epochs=40, lr=0.01, momentum=0.9, batch_size=128,
    seed=42, validation_size=5_000, lr_step=30, lr_gamma=0.1,
)



In [ ]:
augmentation_transforms = {
    "none": [],
    "crop_flip": [v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip(p=0.5)],
}
train_transforms = v2.Compose([
    v2.ToImage(),
    *augmentation_transforms[config["augmentation"]],
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
test_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


Create a reproducible split and data loaders. Validation uses a separate dataset object so later training augmentation cannot affect it. The test loader is created but is not used during experiment selection.


In [ ]:
seed = config["seed"]
batch_size = config["batch_size"]
validation_size = config["validation_size"]

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_data, test_data = cifar10_datasets(train_transforms, test_transforms)
validation_data = copy.copy(train_data)
validation_data.transform = test_transforms

split_generator = torch.Generator().manual_seed(seed)
indices = torch.randperm(len(train_data), generator=split_generator).tolist()
validation_indices = indices[:validation_size]
train_indices = indices[validation_size:]

train_loader = DataLoader(
    Subset(train_data, train_indices), batch_size=batch_size, shuffle=True,
    num_workers=2, generator=torch.Generator().manual_seed(seed),
)
val_loader = DataLoader(
    Subset(validation_data, validation_indices), batch_size=batch_size,
    shuffle=False, num_workers=2,
)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Training images:   {len(train_loader.dataset):,}")
print(f"Validation images: {len(val_loader.dataset):,}")
print(f"Test images:       {len(test_loader.dataset):,} (held out)")
print(f"Split seed:        {seed}")
images, labels = next(iter(train_loader))
print("One batch of images [N, C, H, W]:", tuple(images.shape))


Example training images, after the transforms (any augmentation you add is visible here).

In [ ]:
show_batch(train_loader, CIFAR10_CLASSES)

## 3. CNN Model

To define a neural network in PyTorch, we create a class that inherits ([see python inheritance](https://www.w3schools.com/python/python_inheritance.asp))
from
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
We define the layers of the network in the `__init__` function and
specify how data will pass through the network in the `forward`
function. To accelerate operations in the neural network, we move it to
the GPU.

A simple CNN is defined using [2D convolution](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) and [2D max pooling layers](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html). [Relu activation](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html) is applied after the convolution layers. The output of the last convolution is [flattened](https://pytorch.org/docs/stable/generated/torch.flatten.html) and [2 linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) (dense) layers applied. Only activation is applied after the first linear layer. The network outputs raw scores (logits) with no [softmax activation](https://pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html): the loss function used in the next section (`nn.CrossEntropyLoss`) applies softmax itself.

* **You do not need to modify the network architecture**. You can add regularisation layers and adjust the optimisation/learning rate schedule which is a further method of regularisation in later code cells.

In [ ]:
device = get_device()

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.simple_cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),                      # 32x32 -> 16x16
            nn.Conv2d(64, 64, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),                      # 16x16 -> 8x8
            nn.Conv2d(64, 128, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),                      # 8x8 -> 4x4
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Linear(512, 10),                   # one output (logit) per class
        )

    def forward(self, x):
        logits = self.simple_cnn(x)
        return logits

In [ ]:
# Layer-by-layer summary: output shapes and number of parameters
summary(NeuralNetwork(), input_size=(batch_size, 3, 32, 32))

## 4. Optimisation & Training Loop

The baseline uses cross entropy and SGD. Each experiment starts with a new untrained model. Training is monitored on the validation subset; the separate test set is reserved for the final chosen approach. Batch schedulers step after each batch and epoch schedulers step after each epoch.

The split and random seeds are fixed for repeatable comparisons, although exact GPU results can still vary. Interrupting a run leaves completed epochs in `history`.


In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device, scheduler=None):
    """Train the model for one pass over the data. Returns (average loss, accuracy %)."""
    model.train()                            # training mode: dropout/batch-norm active
    total_loss, correct, seen = 0.0, 0, 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)              # 1. forward pass: predict
        loss = loss_fn(outputs, labels)      # 2. measure how wrong the predictions are
        optimizer.zero_grad()                # 3. clear gradients from the previous step
        loss.backward()                      # 4. compute new gradients
        optimizer.step()                     # 5. update the weights

        if scheduler is not None:            # per-batch schedulers (OneCycleLR, CyclicLR)
            scheduler.step()

        total_loss += loss.item() * len(labels)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        seen += len(labels)

    return total_loss / seen, 100 * correct / seen

Optional: run the next cell **before** training to watch progress live in TensorBoard. TensorBoard is not saved with the notebook, so it will not be visible on GitHub; the plots after training are. (On a local runtime, if the TensorBoard panel stays blank, open [http://localhost:6006](http://localhost:6006) in a new browser tab instead.)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

### 4.1 Experiment settings

The experiment configuration is defined in section 2, before the data loaders are built.


In [ ]:
print(config)


### 4.2 Run an experiment

In [ ]:
# Model, loss and optimiser: created fresh every time this cell runs
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
train_loader.generator.manual_seed(seed)
model = NeuralNetwork().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=config["lr"], momentum=config["momentum"])

# Learning-rate schedulers (optional)
batch_scheduler = None   # stepped after every batch, e.g. OneCycleLR, CyclicLR
epoch_scheduler = (torch.optim.lr_scheduler.StepLR(optimizer, step_size=config["lr_step"],
                                                    gamma=config["lr_gamma"])
                   if "lr_step" in config else None)

history = History(config["run_name"], config=config)
best_weights = None

for epoch in range(config["n_epochs"]):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer,
                                            device, batch_scheduler)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
    history.log(epoch, train_loss, train_acc, val_loss, val_acc,
                lr=optimizer.param_groups[0]["lr"])

    if history.is_best():
        best_weights = copy.deepcopy(model.state_dict())

    if isinstance(epoch_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
        epoch_scheduler.step(val_loss)       # this scheduler needs the validation loss
    elif epoch_scheduler is not None:
        epoch_scheduler.step()

if best_weights is not None:
    model.load_state_dict(best_weights)
    print(f"Restored best validation model from epoch {history.best_index() + 1}.")
print("Finished Training")

### 4.3 Training curves
These plots are saved in the notebook: make sure your submitted notebook shows them for your best run.

In [ ]:
history.plot()
history.summary();

## 5. Evaluate on the validation set

Inspect the best validation accuracy checkpoint on the validation subset while comparing experiments. The separate test set is reserved for final evaluation once the approach is selected.


In [ ]:
y_true, y_pred = predict(model, val_loader, device)
print(classification_report(y_true, y_pred, target_names=CIFAR10_CLASSES))

In [ ]:
plot_confusion_matrix(y_true, y_pred, CIFAR10_CLASSES)

In [ ]:
show_predictions(model, val_loader, device, CIFAR10_CLASSES)

## Notes

### Original baseline

**Changes:** Supplied model and settings, with no augmentation or added regularisation. Used all 50,000 training images and monitored the test set.

**Findings:** The widening gap between training and test performance showed overfitting.

**Result:**
- Final training accuracy: 98.2%
- Final test accuracy: 76.5%
- Best test accuracy: 77.5% at epoch 10

### Validation baseline

**Changes:** Set aside 5,000 training images for validation; otherwise kept the baseline settings.

**Findings:** Validation loss was lowest at epoch 10, then rose as training continued. The model was overfitting. This validation result and the original test result use different splits.

**Result:**
- Final training accuracy: 97.9%
- Final and best validation accuracy: 78.8%

### Experiment 1: crop and flip

**Changes:** Added random crops and horizontal flips to training images.

**Findings:** The extra variation likely helped the model generalise. Training accuracy was measured on augmented images.

**Result:**
- Final training accuracy: 84.1%
- Final validation accuracy: 83.2%
- Final validation loss: 0.4880

### Experiment 2: 40 epochs

**Changes:** Kept crop and flip and trained for 40 epochs instead of 20.

**Findings:** The model benefited from more training, although validation loss rose again after epoch 32.

**Result:**
- Best validation accuracy: 87.3% at epoch 32
- Final validation accuracy: 87.0%
- Final training accuracy: 93.3%
- Lowest validation loss: 0.3747 at epoch 32

### Experiment 3: learning rate schedule

**Changes:** Kept the 40 epoch setup and reduced the learning rate from 0.01 to 0.001 after epoch 30. The best validation checkpoint is restored for evaluation.

**Findings:** Pending.

**Result:** Pending.
